In [ ]:
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
modulePath = os.path.abspath(os.path.join('../../'))
if modulePath not in sys.path:
    sys.path.append(modulePath)
from common.unitConverter import UnitConverter as uc

plt.rcParams.update({
    "font.family" : "serif",
    "font.size" : 18,
    "mathtext.fontset" : "stix",
    "font.serif" : ['STIXGeneral']
})


In [ ]:
folderVec = [
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx/probes/probe1",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_noGeoUpdate/probes/probe1",

    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_32dx_larger_noGeoUpdate/probes/probe1",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_rotating_cylinder/bgk_3d_u001_k05_40dx_larger_noGeoUpdate/probes/probe1",

]
DPhy = 1.0

DLbVec = [
    32,
    32,
    32,
    40,
]

gamma = 1.0

rhoPhyVec = [
    1.204,
    1.204,
    1.204,
    1.204,
]

URefLBVec = [
    0.01,
    0.01,
    0.01,
    0.01,
]

linestyleVec = [
    '-',
    '--',
    ':',
    '-.'
]

labelVec = [
    "Present LBM IBB+MovBody D32 L10",
    "Present LBM IBB D32 L10",
    "Present LBM IBB D32 L25",
    "Present LBM IBB D40 L25",
]

refFolder = "../ref"

tStarVec = [
    1.0,
    2.0,
    3.0,
    4.0,
    5.0,
    6.0,
    8.0
]

markerVec = [
    ['o', 'full'],
    ['o', 'none'],
    ['s', 'full'],
    ['s', 'none'],
    ['d', 'full'],
    ['d', 'none'],
    ['^', 'full']
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    "#FFC629",
    "#119100",
    "#B700FF",
    "#A17808",
]

dxVec = [ DPhy/DLb for DLb in DLbVec ]
unitConverterVec = [ uc(dx, rhoPhy, gamma) for dx, rhoPhy in zip(dxVec, rhoPhyVec) ]


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def get_probe_coords(filePath):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordsStr = header.split('(')[1].split(')')[0].split(',')
        x = float(coordsStr[0])
        y = float(coordsStr[1])
        z = float(coordsStr[2])
    return x,y,z

def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def tStar_to_t(tStar, uRefLB, rLB):
    return tStar * rLB / uRefLB

def t_to_tStar(step, uRefLB, rLB):
    return uRefLB * step / rLB

def x_to_xStar(xLB, rLB):
    return xLB / rLB

def u_to_uStar(uLB, uRefLB):
    return uLB / uRefLB

In [ ]:
for tt in range(0, len(tStarVec)):
    step = tStar_to_t(tStarVec[tt], URefLBVec[0], DLbVec[0] * 0.5)
    print(f"t*={tStarVec[tt]}, step={step}")

In [ ]:

stepVec = []
xStarVecVec = []
uStarVecVecVec = []
vStarVecVecVec = []

for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/probe*.txt"))

    xStarVec = []
    uStarVecVec = [[] for _ in range(len(tStarVec))]
    vStarVecVec = [[] for _ in range(len(tStarVec))]
    for i in range(0, len(fileList)):
        x, y, z = get_probe_coords(fileList[i])
        xLB = unitConverterVec[case].phys_to_lb_length(x)
        yLB = unitConverterVec[case].phys_to_lb_length(y)
        xStarVec.append(x_to_xStar(xLB, DLbVec[case] * 0.5))

        stepCol = read_col_probe(fileList[i], 0, 2)
        u = read_col_probe(fileList[i], 3, 2)
        v = read_col_probe(fileList[i], 4, 2)

        for tt in range(0, len(tStarVec)):
            step = tStar_to_t(tStarVec[tt], URefLBVec[case], DLbVec[case] * 0.5)
            idx = int(np.abs(stepCol - step).argmin())
            uStar = u_to_uStar(u[idx], URefLBVec[case])
            vStar = u_to_uStar(v[idx], URefLBVec[case])

            uStarVecVec[tt].append(uStar)
            vStarVecVec[tt].append(vStar)

    xStarVecVec.append(xStarVec)
    uStarVecVecVec.append(uStarVecVec)
    vStarVecVecVec.append(vStarVecVec)

# print(xStarVecVec)
# print(uStarVecVec)



In [ ]:
refUxVec = []
refUuVec = []
for tt in range(0, len(tStarVec)):
    file = f"{refFolder}/u-x_t{int(tStarVec[tt])}.csv"
    refUxVec.append(read_csv_col(file, 0, 1))
    refUuVec.append(read_csv_col(file, 1, 1))

refVxVec = []
refVvVec = []
for tt in range(0, len(tStarVec)):
    file = f"{refFolder}/v-x_t{int(tStarVec[tt])}.csv"
    if tt == 0:
        refVxVec.append(0)
        refVvVec.append(0)
    else:
        refVxVec.append(read_csv_col(file, 0, 1))
        refVvVec.append(read_csv_col(file, 1, 1))

In [ ]:

figU, axU = plt.subplots(1, 1, figsize=(12, 12), facecolor='w', edgecolor='w')
figV, axV = plt.subplots(1, 1, figsize=(12, 12), facecolor='w', edgecolor='w')
for case in range(0, len(folderVec)):
    for tt in range(0, len(tStarVec)):
        axU.plot(xStarVecVec[case], uStarVecVecVec[case][tt], c=colorVec[tt], lw=1.5, linestyle=linestyleVec[case])
for tt in range(0, len(tStarVec)):
    axU.plot(refUxVec[tt], refUuVec[tt], lw=0, marker=markerVec[tt][0], fillstyle=markerVec[tt][1], c=colorVec[tt], ms=8, label=f"t*={tStarVec[tt]:.1f} (exp., Coutanceau & Menard)")

for case in range(0, len(folderVec)):
    for tt in range(1, len(tStarVec)):
        axV.plot(xStarVecVec[case], vStarVecVecVec[case][tt], c=colorVec[tt], lw=1.5, linestyle=linestyleVec[case])

for tt in range(1, len(tStarVec)):
    axV.plot(refVxVec[tt], refVvVec[tt], lw=0, marker=markerVec[tt][0], fillstyle=markerVec[tt][1], c=colorVec[tt], ms=8, label=f"t*={tStarVec[tt]:.1f} (exp., Coutanceau & Menard)")

for case in range(0, len(folderVec)):
    axU.plot([], [], c='grey', lw=1.5, linestyle=linestyleVec[case], label=labelVec[case])
    axV.plot([], [], c='grey', lw=1.5, linestyle=linestyleVec[case], label=labelVec[case])

axU.set_xlim(1, 6)
axU.set_ylim(-0.5, 1.0)
axU.set_xlabel(r"$x*$")
axU.set_ylabel(r"$u*$")

axV.set_xlim(1, 6)
axV.set_ylim(-0.6, 1.0)
axV.set_xlabel(r"$x*$")
axV.set_ylabel(r"$v*$")

axU.tick_params(width=2.0, axis='both', direction='in')
for spine in axU.spines.values():
    spine.set_linewidth(2.0)

axV.tick_params(width=2.0, axis='both', direction='in')
for spine in axV.spines.values():
    spine.set_linewidth(2.0)

axU.legend(loc='lower right', frameon=False, fontsize=15)
axV.legend(loc='upper right', frameon=False, fontsize=14)